# NB07b — Robot Access Index (Ra)

Builds the **Ra (Robot Accessibility Index)** for each H3 hex cell using
OpenStreetMap walking-network data, then joins back to all 50 Seongnam dongs.

| Component | Weight | Description |
|-----------|--------|-------------|
| `robot_walk_ratio` | 0.40 | Robot-suitable edge length ÷ total walk length |
| `has_crossing`     | 0.30 | ≥ 1 crossing within 200 m buffer |
| `walk_density`     | 0.30 | Total walk edge length within buffer |

**Ra = 0.4 × robot_walk_ratio_norm + 0.3 × has_crossing + 0.3 × walk_density_norm**

> `score_robot = Ra`  (continuous 0–1; NB09 uses Ra directly, not a binary filter)

**Thresholds (for interpretation only)**
- `Ra ≥ 0.3` : 로봇 배송 가능성이 있는 구역
- `Ra ≥ 0.5` : 로봇 배송 적합성이 높은 구역

**Outputs**
- `processed/robot_access_grid.csv` — per-H3 Ra + demand context
- `processed/robot_access_grid.gpkg` — spatial version
- `processed/robot_access_dong_summary.csv` — all 50 dongs
- `processed/robot_access_map.html` — interactive choropleth
- `processed/robot_access_chart.png` — bar + distribution chart

**Requires** `osmnx` + `geopandas`, OR a cached walk network at
`processed/seongnam_walk_network.gpkg`.
The notebook stops with a clear error if neither is available.

In [1]:
import pandas as pd
import numpy as np
import warnings
import os
from pathlib import Path

warnings.filterwarnings('ignore')

BASE       = Path('..').resolve()
PROCESSED  = BASE / 'processed'
CACHE_PATH = PROCESSED / 'seongnam_walk_network.gpkg'

print('Project root :', BASE)
print('Processed dir:', PROCESSED)
print('Cache path   :', CACHE_PATH)


Project root : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset
Processed dir: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset\processed
Cache path   : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset\processed\seongnam_walk_network.gpkg


In [2]:
HAS_OSM = False
try:
    import osmnx as ox
    import geopandas as gpd
    from shapely.geometry import Point
    HAS_OSM = True
    print(f'osmnx {ox.__version__} and geopandas {gpd.__version__} loaded.')
except ImportError as e:
    print(f'osmnx/geopandas not available: {e}')

HAS_CACHE = CACHE_PATH.exists()
print(f'Walk network cache exists: {HAS_CACHE}  ({CACHE_PATH})')

if not HAS_OSM and not HAS_CACHE:
    raise RuntimeError(
        '\n'
        'NB07b cannot proceed: neither osmnx nor the walk-network cache is available.\n'
        '\n'
        'To fix, choose one of:\n'
        '  A) Install osmnx:  conda install -c conda-forge osmnx\n'
        '     Then re-run — it will download and cache the network automatically.\n'
        '\n'
        '  B) Copy a pre-built cache to:\n'
        f'     {CACHE_PATH}\n'
        "     (a .gpkg file with 'edges' and 'nodes' layers from a prior run)\n"
        '\n'
        'Do NOT set Ra=0 for all cells — that produces meaningless scores.'
    )

print('\nDependency check passed — continuing.')


osmnx 2.1.0 and geopandas 1.1.3 loaded.
Walk network cache exists: True  (C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset\processed\seongnam_walk_network.gpkg)

Dependency check passed — continuing.


In [3]:
import geopandas as gpd

demand_gpkg = PROCESSED / 'demand_grid.gpkg'
demand_csv  = PROCESSED / 'demand_grid.csv'

if not demand_gpkg.exists():
    raise FileNotFoundError(
        f'demand_grid.gpkg not found at {demand_gpkg}. '
        'Run NB08_demand_grid_index.ipynb first.'
    )

hex_gdf = gpd.read_file(demand_gpkg)
print(f'H3 grid from GPKG: {len(hex_gdf)} cells, CRS: {hex_gdf.crs}')

if 'h3_index' not in hex_gdf.columns and 'h3_id' in hex_gdf.columns:
    hex_gdf['h3_index'] = hex_gdf['h3_id']

# Pull full column set from CSV (GPKG may omit lowercase aliases due to OGR limits)
DEMAND_COLS = [
    'h3_index', 'lat', 'lon', 'CSV_ADMI_CD', 'ADM_NM', 'GU_NM',
    'Ds', 'demand_grade',
]
if demand_csv.exists():
    dcsv = pd.read_csv(demand_csv, encoding='utf-8-sig',
                       usecols=lambda c: c in DEMAND_COLS)
    before = set(hex_gdf.columns)
    hex_gdf = hex_gdf.merge(dcsv, on='h3_index', how='left', suffixes=('', '_csv'))
    # Prefer CSV value when GPKG column is missing or all-null
    for col in DEMAND_COLS:
        csv_col = f'{col}_csv'
        if csv_col in hex_gdf.columns:
            if col not in hex_gdf.columns:
                hex_gdf[col] = hex_gdf[csv_col]
            else:
                hex_gdf[col] = hex_gdf[col].combine_first(hex_gdf[csv_col])
            hex_gdf.drop(columns=[csv_col], inplace=True)
    added = set(hex_gdf.columns) - before
    print(f'  Merged from CSV: {sorted(added)}')

print(f'lat/lon available  : {hex_gdf["lat"].notna().sum()}/{len(hex_gdf)}')
print(f'Ds available       : {hex_gdf["Ds"].notna().sum()}/{len(hex_gdf)}')
print(f'demand_grade avail : {hex_gdf["demand_grade"].notna().sum()}/{len(hex_gdf)}')
print(f'ADM_NM sample      : {hex_gdf["ADM_NM"].dropna().head(3).tolist()}')


H3 grid from GPKG: 283 cells, CRS: EPSG:4326
  Merged from CSV: []
lat/lon available  : 283/283
Ds available       : 283/283
demand_grade avail : 283/283
ADM_NM sample      : ['은행2동', '삼평동', '도촌동']


In [4]:
if HAS_CACHE:
    print(f'Loading cached walk network from {CACHE_PATH} ...')
    edges = gpd.read_file(CACHE_PATH, layer='edges')
    nodes = gpd.read_file(CACHE_PATH, layer='nodes')
    print(f'  edges: {len(edges):,}  nodes: {len(nodes):,}  CRS: {edges.crs}')
    DATA_SOURCE = 'cache'
else:
    print('Downloading Seongnam walking network from OSM (1-3 min first run) ...')
    ox.settings.log_console = False
    G = ox.graph_from_place(
        'Seongnam-si, Gyeonggi-do, South Korea',
        network_type='walk',
        retain_all=False,
        truncate_by_edge=True,
    )
    nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)
    edges = edges.reset_index().to_crs('EPSG:4326')
    nodes = nodes.reset_index().to_crs('EPSG:4326')
    print(f'  Downloaded: edges={len(edges):,}, nodes={len(nodes):,}')
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    edges.to_file(CACHE_PATH, layer='edges', driver='GPKG')
    nodes.to_file(CACHE_PATH, layer='nodes', driver='GPKG')
    print(f'  Cached to {CACHE_PATH}')
    DATA_SOURCE = 'osm_download'

print(f'\nData source: {DATA_SOURCE}')
if 'highway' in edges.columns:
    hw = edges['highway'].apply(lambda h: h[0] if isinstance(h, list) else str(h))
    print('Highway types (top 10):')
    print(hw.value_counts().head(10).to_string())


Loading cached walk network from C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset\processed\seongnam_walk_network.gpkg ...


  edges: 54,840  nodes: 19,271  CRS: EPSG:4326

Data source: cache
Highway types (top 10):
highway
service           17734
residential       13376
footway           11372
tertiary           3474
secondary          2074
path               1532
unclassified       1304
primary             930
secondary_link      542
living_street       362


In [5]:
ROBOT_WEIGHTS = {
    'footway'      : 1.0,
    'pedestrian'   : 1.0,
    'living_street': 0.9,
    'crossing'     : 0.9,
    'cycleway'     : 0.8,
    'path'         : 0.7,
    'service'      : 0.6,
    'residential'  : 0.5,
    'unclassified' : 0.4,
}
DEFAULT_WEIGHT = 0.3

def _extract_highway(val):
    if isinstance(val, list):
        return val[0] if val else 'unknown'
    return str(val)

def _is_crossing_tag(val, accept):
    '''Return True if val (possibly a list) contains any value in accept.'''
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return False
    items = val if isinstance(val, list) else [val]
    return any(str(v).strip().lower() in accept for v in items)

edges['highway_clean'] = edges['highway'].apply(_extract_highway)
edges['robot_weight']  = edges['highway_clean'].map(ROBOT_WEIGHTS).fillna(DEFAULT_WEIGHT)

if 'length' in edges.columns:
    edges['length_m'] = pd.to_numeric(edges['length'], errors='coerce').fillna(0)
else:
    edges_proj_tmp = edges.to_crs('EPSG:5179')
    edges['length_m'] = edges_proj_tmp.geometry.length

# ── Crossing extraction ───────────────────────────────────────────────────────
# OSM encodes crossings primarily as NODES (highway=crossing), not edges.
# osmnx populates them in the `nodes` GeoDataFrame.
# Edge-level footway=crossing is a secondary source (the traversal path).
POSITIVE_CROSSING_VALS = {
    'marked', 'uncontrolled', 'traffic_signals', 'yes',
    'zebra', 'pelican', 'toucan', 'pegasus', 'supervised',
}

node_cross_mask = pd.Series(False, index=nodes.index)
if 'highway' in nodes.columns:
    node_cross_mask |= nodes['highway'].apply(
        lambda h: _is_crossing_tag(h, {'crossing'})
    )
if 'crossing' in nodes.columns:
    node_cross_mask |= nodes['crossing'].apply(
        lambda c: _is_crossing_tag(c, POSITIVE_CROSSING_VALS)
    )
crossing_nodes = nodes[node_cross_mask].copy()

edge_cross_mask = edges['highway_clean'] == 'crossing'
if 'footway' in edges.columns:
    edge_cross_mask |= edges['footway'].apply(
        lambda f: _is_crossing_tag(f, {'crossing'})
    )
crossing_edges = edges[edge_cross_mask].copy()

CROSSING_AVAILABLE = len(crossing_nodes) + len(crossing_edges) > 0

print(f'Total edges                       : {len(edges):,}')
print(f'Total nodes                       : {len(nodes):,}')
print(f'Robot-suitable edges (>=0.5)      : {(edges["robot_weight"] >= 0.5).sum():,}')
print(f'Crossing NODES (highway=crossing) : {len(crossing_nodes):,}')
print(f'Crossing EDGES (footway=crossing) : {len(crossing_edges):,}')
print(f'Crossing data available           : {CROSSING_AVAILABLE}')
if not CROSSING_AVAILABLE:
    print()
    print('WARNING: No crossing data found in OSM network.')
    print('  has_crossing will be 0 for all cells.')
    print('  Ra will use only robot_walk_ratio and walk_density components.')
print()
print('Highway distribution (edges):')
print(edges['highway_clean'].value_counts().head(12).to_string())


Total edges                       : 54,840
Total nodes                       : 19,271
Robot-suitable edges (>=0.5)      : 44,582
Crossing NODES (highway=crossing) : 419
Crossing EDGES (footway=crossing) : 0
Crossing data available           : True

Highway distribution (edges):
highway_clean
service           17734
residential       13376
footway           11372
tertiary           3474
secondary          2074
path               1532
unclassified       1304
primary             930
secondary_link      542
living_street       362
primary_link        292
trunk_link          288


In [6]:
from shapely.geometry import Point

EPSG_KR  = 'EPSG:5179'
BUFFER_M = 200   # H3 res-8 hex radius ~261 m

edges_proj          = edges.to_crs(EPSG_KR)
crossing_nodes_proj = crossing_nodes.to_crs(EPSG_KR) if len(crossing_nodes) > 0 else crossing_nodes
crossing_edges_proj = crossing_edges.to_crs(EPSG_KR) if len(crossing_edges) > 0 else crossing_edges

# Combine: node Points + edge LineStrings → single list, single STRtree
cross_geom_all = []
if len(crossing_nodes_proj) > 0:
    cross_geom_all.extend(list(crossing_nodes_proj.geometry))
if len(crossing_edges_proj) > 0:
    cross_geom_all.extend(list(crossing_edges_proj.geometry))

print(f'Combined crossing geometries : {len(cross_geom_all):,}')
print(f'  from nodes : {len(crossing_nodes_proj):,}')
print(f'  from edges : {len(crossing_edges_proj):,}')

cell_gdf = gpd.GeoDataFrame(
    hex_gdf[['h3_index', 'lat', 'lon', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD']].copy(),
    geometry=[Point(row.lon, row.lat) for row in hex_gdf.itertuples()],
    crs='EPSG:4326'
)
cell_proj = cell_gdf.to_crs(EPSG_KR)

print(f'\nCells to process : {len(cell_proj)}')
print(f'Buffer radius    : {BUFFER_M} m')
print(f'CRS              : {EPSG_KR}')


Combined crossing geometries : 419
  from nodes : 419
  from edges : 0

Cells to process : 283
Buffer radius    : 200 m
CRS              : EPSG:5179


In [7]:
from shapely.strtree import STRtree

edges_geom = list(edges_proj.geometry)
edges_tree = STRtree(edges_geom)
cross_tree = STRtree(cross_geom_all) if cross_geom_all else None

ra_records = []
n = len(cell_proj)

for idx, (_, cell) in enumerate(cell_proj.iterrows()):
    buf = cell.geometry.buffer(BUFFER_M)

    cand = edges_tree.query(buf)
    hits = [i for i in cand if edges_geom[i].intersects(buf)]

    if not hits:
        ra_records.append({
            'h3_index'        : cell['h3_index'],
            'total_walk_length': 0.0,
            'robot_walk_length': 0.0,
            'n_crossings'     : 0,
            'avg_robot_weight': 0.0,
            'ratio_footway'   : 0.0,
            'has_crossing'    : 0.0,
            'walk_density_raw': 0.0,
        })
    else:
        sub          = edges_proj.iloc[hits]
        total_length = float(sub['length_m'].sum())
        robot_length = float((sub['length_m'] * sub['robot_weight']).sum())
        avg_weight   = float(sub['robot_weight'].mean())

        if cross_tree is not None:
            cross_cand = cross_tree.query(buf)
            n_cross = sum(1 for i in cross_cand if cross_geom_all[i].intersects(buf))
        else:
            n_cross = 0

        ra_records.append({
            'h3_index'        : cell['h3_index'],
            'total_walk_length': round(total_length, 1),
            'robot_walk_length': round(robot_length, 1),
            'n_crossings'     : int(n_cross),
            'avg_robot_weight': round(avg_weight, 4),
            'ratio_footway'   : round(robot_length / max(total_length, 1), 4),
            'has_crossing'    : 1.0 if n_cross > 0 else 0.0,
            'walk_density_raw': total_length,
        })

    if (idx + 1) % 50 == 0 or idx + 1 == n:
        print(f'  {idx+1}/{n} processed', end='\r')

print(f'\nDone. {len(ra_records)} records.')
ra_df = pd.DataFrame(ra_records)
print(ra_df[['ratio_footway', 'has_crossing', 'walk_density_raw']].describe().round(3))


  283/283 processed
Done. 283 records.
       ratio_footway  has_crossing  walk_density_raw
count        283.000       283.000           283.000
mean           0.500         0.110          6614.390
std            0.213         0.313          5650.173
min            0.000         0.000             0.000
25%            0.386         0.000          2337.244
50%            0.531         0.000          5560.495
75%            0.679         0.000          9405.456
max            0.932         1.000         30416.576


In [8]:
P_LO, P_HI = 1, 99

def clip_minmax(series: pd.Series, p_lo=P_LO, p_hi=P_HI) -> pd.Series:
    valid = series.dropna()
    if valid.empty:
        return pd.Series(0.0, index=series.index)
    lo, hi = valid.quantile(p_lo / 100), valid.quantile(p_hi / 100)
    rng = hi - lo
    if rng == 0:
        return pd.Series(0.5, index=series.index)
    return ((series.clip(lo, hi) - lo) / rng).fillna(0.0)

ra_df['ratio_footway_norm'] = clip_minmax(ra_df['ratio_footway'])
ra_df['walk_density_norm']  = clip_minmax(ra_df['walk_density_raw'])

ra_df['Ra'] = (
    0.4 * ra_df['ratio_footway_norm']
  + 0.3 * ra_df['has_crossing']
  + 0.3 * ra_df['walk_density_norm']
)

# Crossing contribution report
cells_with_crossing = int((ra_df['has_crossing'] > 0).sum())
total_detected = int(ra_df['n_crossings'].sum())
print('[Crossing contribution]')
print(f'  Total detected crossings (sum over all cells): {total_detected:,}')
print(f'  H3 cells with crossing access                : {cells_with_crossing} / {len(ra_df)}')
if not CROSSING_AVAILABLE:
    print('  WARNING: has_crossing = 0 for all cells (no crossing data in OSM network).')
    print('           Ra is computed from robot_walk_ratio and walk_density only.')
else:
    cross_contrib = 0.3 * ra_df['has_crossing'].mean()
    print(f'  Average has_crossing contribution to Ra     : {cross_contrib:.4f}')
print()
print('[Ra statistics]')
print(ra_df['Ra'].describe().round(4).to_string())


[Crossing contribution]
  Total detected crossings (sum over all cells): 102
  H3 cells with crossing access                : 31 / 283
  Average has_crossing contribution to Ra     : 0.0329

[Ra statistics]
count    283.0000
mean       0.3380
std        0.1849
min        0.0000
25%        0.2551
50%        0.3343
75%        0.3819
max        0.8773


In [9]:
ra_df['Ra_percentile'] = ra_df['Ra'].rank(pct=True) * 100

def _grade(pct):
    if pct >= 80: return 'A'
    elif pct >= 50: return 'B'
    elif pct >= 20: return 'C'
    else:           return 'D'

ra_df['Ra_grade'] = ra_df['Ra_percentile'].apply(_grade)
print('Grade distribution:')
print(ra_df['Ra_grade'].value_counts().sort_index().to_string())


Grade distribution:
Ra_grade
A    57
B    85
C    85
D    56


In [10]:
# Merge Ra results back into hex_gdf which already carries lat/lon/Ds/demand_grade
grid_ra = hex_gdf.merge(
    ra_df[['h3_index', 'Ra', 'Ra_grade', 'Ra_percentile',
           'total_walk_length', 'robot_walk_length', 'n_crossings',
           'avg_robot_weight', 'ratio_footway', 'has_crossing',
           'ratio_footway_norm', 'walk_density_norm']],
    on='h3_index', how='left'
)

# Fill unmatched cells conservatively
for col in ['Ra', 'Ra_percentile', 'ratio_footway_norm', 'walk_density_norm',
            'total_walk_length', 'robot_walk_length', 'ratio_footway',
            'has_crossing', 'avg_robot_weight']:
    if col in grid_ra.columns:
        grid_ra[col] = grid_ra[col].fillna(0.0)

grid_ra['Ra_grade']    = grid_ra['Ra_grade'].fillna('D')
grid_ra['n_crossings'] = grid_ra['n_crossings'].fillna(0).astype(int)

print(f'grid_ra shape: {grid_ra.shape}')
print(f'Ra range     : {grid_ra["Ra"].min():.3f} - {grid_ra["Ra"].max():.3f}')
print(f'Ds available : {grid_ra["Ds"].notna().sum()}/{len(grid_ra)}')


grid_ra shape: (283, 45)


Ra range     : 0.000 - 0.877
Ds available : 283/283


In [11]:
# Aggregate Ra per dong from H3 cells
h3_agg = (
    grid_ra.groupby('CSV_ADMI_CD')
    .agg(
        avg_Ra               = ('Ra',           'mean'),
        max_Ra               = ('Ra',           'max'),
        median_Ra            = ('Ra',           'median'),
        robot_possible_cells = ('Ra',           lambda x: (x >= 0.3).sum()),
        robot_good_cells     = ('Ra',           lambda x: (x >= 0.5).sum()),
        total_cells          = ('Ra',           'count'),
        avg_n_crossings      = ('n_crossings',  'mean'),
    )
    .reset_index()
)

# Load full 50-dong boundary to ensure all dongs appear in summary
bnd = gpd.read_file(PROCESSED / 'seongnam_boundary.gpkg', layer='dong')
bnd_tbl = bnd[['ADM_NM', 'GU_NM', 'CSV_ADMI_CD']].copy()

# Left join: boundary (50 dongs) <- H3 aggregation
# Dongs without H3 cells get NaN for metric cols
dong_summary = bnd_tbl.merge(h3_agg, on='CSV_ADMI_CD', how='left')

dong_summary['total_cells'] = dong_summary['total_cells'].fillna(0).astype(int)
dong_summary['robot_possible_cells'] = dong_summary['robot_possible_cells'].fillna(0).astype(int)
dong_summary['robot_good_cells']     = dong_summary['robot_good_cells'].fillna(0).astype(int)
dong_summary['robot_possible_ratio'] = np.where(
    dong_summary['total_cells'] > 0,
    dong_summary['robot_possible_cells'] / dong_summary['total_cells'],
    np.nan
)
dong_summary['robot_good_ratio'] = np.where(
    dong_summary['total_cells'] > 0,
    dong_summary['robot_good_cells'] / dong_summary['total_cells'],
    np.nan
)
dong_summary['robot_access_note'] = np.where(
    dong_summary['total_cells'] == 0,
    'no H3 cells — boundary only',
    'aggregated from H3 cells'
)

dong_summary = dong_summary.sort_values('avg_Ra', ascending=False, na_position='last').reset_index(drop=True)

DONG_COLS = [
    'ADM_NM', 'GU_NM', 'CSV_ADMI_CD',
    'avg_Ra', 'max_Ra', 'median_Ra',
    'robot_possible_cells', 'robot_good_cells', 'total_cells',
    'robot_possible_ratio', 'robot_good_ratio',
    'avg_n_crossings', 'robot_access_note',
]
dong_summary = dong_summary[DONG_COLS]

print(f'Dong summary rows: {len(dong_summary)} (expect 50)')
print(f'Dongs with no H3 cells: {(dong_summary["total_cells"] == 0).sum()}')
print()
print('Top 10 by avg_Ra:')
show = ['ADM_NM', 'GU_NM', 'avg_Ra', 'robot_good_cells', 'total_cells', 'robot_good_ratio']
print(dong_summary.head(10)[show].round(4).to_string(index=False))


Dong summary rows: 50 (expect 50)
Dongs with no H3 cells: 6

Top 10 by avg_Ra:
ADM_NM GU_NM  avg_Ra  robot_good_cells  total_cells  robot_good_ratio
  수내2동   분당구  0.8181                 2            2            1.0000
   정자동   분당구  0.8169                 2            2            1.0000
  수내1동   분당구  0.8158                 2            2            1.0000
  야탑1동   분당구  0.7844                 2            2            1.0000
  정자1동   분당구  0.6485                 1            2            0.5000
   백현동   분당구  0.6333                 4            5            0.8000
  야탑2동   분당구  0.5629                 1            1            1.0000
  태평1동   수정구  0.5093                 1            3            0.3333
  정자2동   분당구  0.4775                 0            1            0.0000
  수진2동   수정구  0.4726                 0            1            0.0000


In [12]:
# ── CSV: all required downstream columns ─────────────────────────────────────
CSV_COLS = [
    'h3_index', 'lat', 'lon', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD',
    'Ds', 'demand_grade',
    'Ra', 'Ra_grade', 'Ra_percentile',
    'total_walk_length', 'robot_walk_length', 'n_crossings',
    'avg_robot_weight', 'ratio_footway', 'has_crossing',
    'ratio_footway_norm', 'walk_density_norm',
]
csv_df = grid_ra[[c for c in CSV_COLS if c in grid_ra.columns]].copy()
csv_path = PROCESSED / 'robot_access_grid.csv'
csv_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

# ── GPKG: same columns + geometry; avoid OGR case-insensitive conflict ────────
# 'gu_nm' (lowercase from GPKG) vs 'GU_NM' (alias) conflict — keep only 'GU_NM'
GPKG_COLS = [
    'h3_index', 'lat', 'lon', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD',
    'Ds', 'demand_grade',
    'Ra', 'Ra_grade', 'Ra_percentile',
    'total_walk_length', 'robot_walk_length', 'n_crossings',
    'avg_robot_weight', 'ratio_footway', 'has_crossing',
    'ratio_footway_norm', 'walk_density_norm',
    'geometry',
]
# Drop any lowercase duplicates that would conflict in OGR
gpkg_gdf = grid_ra[[c for c in GPKG_COLS if c in grid_ra.columns]].copy()
for lc in ['gu_nm', 'adm_nm', 'h3_id', 'dong_nm']:
    if lc in gpkg_gdf.columns:
        gpkg_gdf.drop(columns=[lc], inplace=True)
gpkg_gdf['Ra_grade']     = gpkg_gdf['Ra_grade'].astype(str)
gpkg_gdf['demand_grade'] = gpkg_gdf['demand_grade'].astype(str)
gpkg_path = PROCESSED / 'robot_access_grid.gpkg'
gpkg_gdf.to_file(gpkg_path, driver='GPKG')

# ── Dong summary ─────────────────────────────────────────────────────────────
dong_path = PROCESSED / 'robot_access_dong_summary.csv'
dong_summary.to_csv(dong_path, index=False, encoding='utf-8-sig')

for p in [csv_path, gpkg_path, dong_path]:
    print(f'  {p.name}: {p.stat().st_size // 1024} KB  ({p.stat().st_size:,} bytes)')


  robot_access_grid.csv: 58 KB  (59,485 bytes)
  robot_access_grid.gpkg: 200 KB  (204,800 bytes)
  robot_access_dong_summary.csv: 6 KB  (6,913 bytes)


In [13]:
try:
    import folium
    import branca.colormap as cm

    city_centroid = (
        grid_ra.dissolve().to_crs('EPSG:5179').centroid.to_crs('EPSG:4326').iloc[0]
    )
    center = [city_centroid.y, city_centroid.x]

    m = folium.Map(location=center, zoom_start=12, tiles='CartoDB positron')
    cmap = cm.linear.YlOrRd_09.scale(0, 1)
    cmap.caption = 'Robot Access Index (Ra)   |   score_robot = Ra'

    for _, row in grid_ra.iterrows():
        if row.geometry is None:
            continue
        coords = [[y, x] for x, y in row.geometry.exterior.coords]
        ra_val = float(row['Ra']) if pd.notna(row['Ra']) else 0.0
        color  = cmap(ra_val)
        ds_str  = f"{row['Ds']:.3f}" if 'Ds' in row and pd.notna(row['Ds']) else 'N/A'
        dg_str  = str(row.get('demand_grade', 'N/A'))
        popup = (
            f"<b>{row.get('ADM_NM', '')} ({row.get('GU_NM', '')})</b><br>"
            f"<hr style='margin:3px 0'>"
            f"<b>Ra</b> = {ra_val:.3f} &nbsp; Grade: {row.get('Ra_grade','?')}<br>"
            f"<b>Ds</b> = {ds_str} &nbsp; Demand grade: {dg_str}<br>"
            f"<hr style='margin:3px 0'>"
            f"Walk length: {row.get('total_walk_length', 0):.0f} m<br>"
            f"Crossings: {int(row.get('n_crossings', 0))}"
        )
        folium.Polygon(
            locations=coords, color=color, fill=True, fill_color=color,
            fill_opacity=0.65, weight=0.4,
            popup=folium.Popup(popup, max_width=260)
        ).add_to(m)

    cmap.add_to(m)
    map_path = PROCESSED / 'robot_access_map.html'
    m.save(str(map_path))
    print(f'Map saved: {map_path.name} ({map_path.stat().st_size // 1024} KB)')
except Exception as e:
    print(f'Map skipped: {e}')


Map saved: robot_access_map.html (434 KB)


In [14]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.font_manager as fm

    plt.rcParams['axes.unicode_minus'] = False
    for fname in ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']:
        try:
            fm.findfont(fname, fallback_to_default=False)
            plt.rcParams['font.family'] = fname
            break
        except Exception:
            pass

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Seongnam Robot Access Index (Ra)   |   score_robot = Ra', fontsize=13)

    # Bar chart: top 15 by avg_Ra (only dongs with at least 1 H3 cell)
    top15 = dong_summary[dong_summary['total_cells'] > 0].head(15)
    axes[0].barh(top15['ADM_NM'][::-1], top15['avg_Ra'][::-1],
                 color='#43A047', alpha=0.85)
    axes[0].set_xlabel('Average Ra')
    axes[0].set_title('Top 15 Dongs by Robot Accessibility')
    axes[0].axvline(0.3, color='orange', ls='--', alpha=0.7,
                    label='Ra=0.3 (possible)')
    axes[0].axvline(0.5, color='red', ls='--', alpha=0.7,
                    label='Ra=0.5 (good)')
    axes[0].legend(fontsize=9)
    axes[0].grid(axis='x', alpha=0.3)

    # Histogram: Ra distribution across H3 cells
    ra_vals = grid_ra['Ra'].dropna()
    axes[1].hist(ra_vals, bins=30, color='#1E88E5', alpha=0.8, edgecolor='white')
    axes[1].axvline(ra_vals.mean(), color='red', ls='-',
                    label=f'Mean {ra_vals.mean():.3f}')
    axes[1].axvline(0.3, color='orange', ls='--', alpha=0.8, label='Ra=0.3')
    axes[1].axvline(0.5, color='red',    ls='--', alpha=0.8, label='Ra=0.5')
    axes[1].set_xlabel('Ra (Robot Access Index)')
    axes[1].set_ylabel('H3 cell count')
    axes[1].set_title('Ra Distribution')
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    chart_path = PROCESSED / 'robot_access_chart.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Chart saved: {chart_path.name} ({chart_path.stat().st_size // 1024} KB)')
except Exception as e:
    print(f'Chart skipped: {e}')


Chart saved: robot_access_chart.png (101 KB)


In [15]:
print('=' * 65)
print('NB07b — Robot Access Index (Ra) — Final Report')
print('=' * 65)
print(f'Data source          : {DATA_SOURCE}')
print()
print('[H3 grid]')
print(f'  Cells processed    : {len(grid_ra)}')
print(f'  Ra = 0 (no access) : {(grid_ra["Ra"] == 0).sum()}')
print(f'  Ra >= 0.3 (possible): {(grid_ra["Ra"] >= 0.3).sum()}')
print(f'  Ra >= 0.5 (good)   : {(grid_ra["Ra"] >= 0.5).sum()}')
print()
print('[Ra statistics]')
print(grid_ra['Ra'].describe().round(4).to_string())
print()
grade_counts = grid_ra['Ra_grade'].value_counts().sort_index()
for g, cnt in grade_counts.items():
    print(f'  Grade {g}: {cnt} cells')
print()
print('[Crossing data]')
total_cross = int(grid_ra['n_crossings'].sum())
cells_cross = int((grid_ra['n_crossings'] > 0).sum())
print(f'  Total crossings detected : {total_cross:,}')
print(f'  Cells with crossing      : {cells_cross} / {len(grid_ra)}')
if not CROSSING_AVAILABLE:
    print('  has_crossing contribution: UNAVAILABLE (no crossing data in OSM network)')
else:
    print(f'  has_crossing contribution: ACTIVE (weight 0.3)')
print()
print('[Dong summary]')
print(f'  Rows (expect 50)         : {len(dong_summary)}')
print(f'  Dongs with no H3 cells   : {(dong_summary["total_cells"] == 0).sum()}')
print()
print('Top 10 dongs by avg_Ra:')
s1 = ['ADM_NM', 'GU_NM', 'avg_Ra', 'robot_good_cells', 'total_cells', 'robot_good_ratio']
print(dong_summary.head(10)[s1].round(4).to_string(index=False))
print()
print('Top 10 dongs by robot_good_ratio:')
s2 = ['ADM_NM', 'GU_NM', 'avg_Ra', 'robot_good_ratio', 'total_cells']
by_ratio = dong_summary[dong_summary['total_cells'] > 0].nlargest(10, 'robot_good_ratio')
print(by_ratio[s2].round(4).to_string(index=False))
print()
print('[Null check — grid]')
for col in ['h3_index', 'lat', 'lon', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD', 'Ds', 'Ra']:
    if col in grid_ra.columns:
        nn = grid_ra[col].isna().sum()
        flag = ' <-- PROBLEM' if nn > 0 and col in ('h3_index', 'lat', 'lon', 'Ra') else ''
        print(f'  {col}: {nn} nulls{flag}')
    else:
        print(f'  {col}: COLUMN MISSING')
print()
print('[Output files]')
for fname in ['robot_access_grid.csv', 'robot_access_grid.gpkg',
              'robot_access_dong_summary.csv', 'robot_access_map.html',
              'robot_access_chart.png']:
    p = PROCESSED / fname
    status = f'{p.stat().st_size // 1024} KB' if p.exists() else 'NOT FOUND'
    print(f'  {fname}: {status}')
print()
print('[Formula]')
print('  Ra = 0.4 x robot_walk_ratio_norm')
print('     + 0.3 x has_crossing')
print('     + 0.3 x walk_density_norm')
print()
print('  score_robot = Ra  (continuous; NB09 uses Ra directly)')


NB07b — Robot Access Index (Ra) — Final Report
Data source          : cache

[H3 grid]
  Cells processed    : 283
  Ra = 0 (no access) : 27
  Ra >= 0.3 (possible): 177
  Ra >= 0.5 (good)   : 35

[Ra statistics]
count    283.0000
mean       0.3380
std        0.1849
min        0.0000
25%        0.2551
50%        0.3343
75%        0.3819
max        0.8773

  Grade A: 57 cells
  Grade B: 85 cells
  Grade C: 85 cells
  Grade D: 56 cells

[Crossing data]
  Total crossings detected : 102
  Cells with crossing      : 31 / 283
  has_crossing contribution: ACTIVE (weight 0.3)

[Dong summary]
  Rows (expect 50)         : 50
  Dongs with no H3 cells   : 6

Top 10 dongs by avg_Ra:
ADM_NM GU_NM  avg_Ra  robot_good_cells  total_cells  robot_good_ratio
  수내2동   분당구  0.8181                 2            2            1.0000
   정자동   분당구  0.8169                 2            2            1.0000
  수내1동   분당구  0.8158                 2            2            1.0000
  야탑1동   분당구  0.7844                 2     